# Auditoría del Grafo de Conocimiento

Diagnóstico de calidad del grafo Neo4j y del fichero `output/extraction_results.json`.

**Secciones:**
1. Estadísticas generales (Neo4j)
2. Fragmentación de personajes clave (Holmes / Watson)
3. Personajes genéricos / ruido
4. Relaciones huérfanas
5. Tipos de relación fuera del schema
6. Cobertura de cadenas de deducción
7. Consistencia de títulos de relatos
8. Distribución de objetos y eventos
9. Resumen ejecutivo (panel de estado)

In [1]:
import json
from collections import Counter
from pathlib import Path

from graphrag.graph.neo4j_manager import Neo4jManager

RESULTS_PATH = Path("../output/extraction_results.json")
with open(RESULTS_PATH, encoding="utf-8") as f:
    data = json.load(f)

neo4j = Neo4jManager()
print(f"Relatos en JSON  : {len(data)}")
print(f"Relatos          : {list(data.keys())}")

# Constantes reutilizadas en secciones posteriores
VALID_REL_TYPES = {
    "APPEARS_IN", "KNOWS", "INVESTIGATES", "USES", "FOUND_AT",
    "PRESENT_IN", "TAKES_PLACE_IN", "FOLLOWS", "BASED_ON", "LEADS_TO",
    "PARTICIPATES_IN", "OCCURS_IN", "MENTIONS", "HAS_CHUNK", "BELONGS_TO", "LIVES_AT",
}
CHAR_REL_TYPES = {
    "APPEARS_IN", "KNOWS", "INVESTIGATES", "USES", "PRESENT_IN", "PARTICIPATES_IN",
}

# Mismo criterio que entity_extractor.py
_GENERIC_PREFIXES = (
    "the ", "a ", "an ",
    "my ", "your ", "his ", "her ",
    "our ", "this ", "that ",
)
_GENERIC_EXACT = frozenset({
    "i", "we", "he", "she", "they", "you", "it",
    "him", "her", "them", "us", "me",
    "doctor", "inspector", "narrator", "gentleman",
    "companion", "stranger", "visitor", "assistant",
    "husband", "wife", "maid", "porter", "lad",
    "coachman", "landlord", "landlady",
})

def is_generic(name: str) -> bool:
    n = " ".join(name.lower().strip().split())
    return any(n.startswith(p) for p in _GENERIC_PREFIXES) or n in _GENERIC_EXACT


Relatos en JSON  : 10
Relatos          : ['Silver Blaze', 'The Final Problem', 'The Adventure Of The Dancing Men', 'A Scandal In Bohemia', 'The Red-Headed League', 'A Case Of Identity', 'The Five Orange Pips', 'The Adventure Of The Blue Carbuncle', 'The Adventure Of The Speckled Band', 'The Adventure Of The Copper Beeches']


## 1. Estadísticas generales del grafo Neo4j

In [2]:
stats = neo4j.get_stats()
print("Nodos por etiqueta:")
for label, count in sorted(stats.items(), key=lambda x: -x[1]):
    print(f"  {label:15s}: {count:5d}")

rel_stats = neo4j.execute_query("""
    MATCH ()-[r]->()
    RETURN type(r) AS rel_type, count(*) AS cnt
    ORDER BY cnt DESC
""")
print("\nRelaciones por tipo en Neo4j:")
for row in rel_stats:
    print(f"  {row['rel_type']:20s}: {row['cnt']:5d}")

Nodos por etiqueta:
  Event          :   722
  Object         :   562
  Chunk          :   350
  Deduction      :   350
  Crime          :   317
  Scene          :   303
  Location       :   261
  Character      :   186
  Story          :    10

Relaciones por tipo en Neo4j:
  MENTIONS            :  1881
  USES                :   422
  HAS_CHUNK           :   350
  BELONGS_TO          :   303
  KNOWS               :   266
  APPEARS_IN          :   223
  BASED_ON            :   202
  FOUND_AT            :   161
  LIVES_AT            :    85
  LEADS_TO            :    39


## 2. Fragmentación de personajes clave

Holmes y Watson deberían ser **1 nodo cada uno** en Neo4j.

In [3]:
holmes_nodes = neo4j.execute_query("""
    MATCH (c:Character)
    WHERE toLower(c.name) CONTAINS 'holmes'
    RETURN c.name AS name, c.aliases AS aliases,
           size([(c)-[r]-() | r]) AS connections
    ORDER BY connections DESC
""")
watson_nodes = neo4j.execute_query("""
    MATCH (c:Character)
    WHERE toLower(c.name) CONTAINS 'watson'
    RETURN c.name AS name, c.aliases AS aliases,
           size([(c)-[r]-() | r]) AS connections
    ORDER BY connections DESC
""")

print(f"Nodos Holmes en Neo4j : {len(holmes_nodes)}  (esperado: 1)")
for n in holmes_nodes:
    print(f"  '{n['name']}' | aliases: {n['aliases']} | conexiones: {n['connections']}")

print(f"\nNodos Watson en Neo4j : {len(watson_nodes)}  (esperado: 1)")
for n in watson_nodes:
    print(f"  '{n['name']}' | aliases: {n['aliases']} | conexiones: {n['connections']}")

print(f"\n[METRICA] Nodos extra por fragmentacion: {(len(holmes_nodes)-1)+(len(watson_nodes)-1)}  (ideal: 0)")

Nodos Holmes en Neo4j : 1  (esperado: 1)
  'Sherlock Holmes' | aliases: ['Holmes', 'MR. HOLMES'] | conexiones: 318

Nodos Watson en Neo4j : 2  (esperado: 1)
  'Dr. Watson' | aliases: ['Watson'] | conexiones: 188
  'Mrs. Watson' | aliases: [] | conexiones: 5

[METRICA] Nodos extra por fragmentacion: 1  (ideal: 0)


In [4]:
print(f"{'Relato':45s}  {'Holmes canonical':32s}  Watson canonical")
print("-" * 100)
for story, result in data.items():
    chars = result["entities"].get("characters", [])
    holmes = next((c["name"] for c in chars if "holmes" in c["name"].lower()), "--")
    watson = next((c["name"] for c in chars if "watson" in c["name"].lower()), "--")
    print(f"  {story:43s}  {holmes:32s}  {watson}")

Relato                                         Holmes canonical                  Watson canonical
----------------------------------------------------------------------------------------------------
  Silver Blaze                                 Sherlock Holmes                   Dr. Watson
  The Final Problem                            Sherlock Holmes                   Dr. Watson
  The Adventure Of The Dancing Men             Sherlock Holmes                   Dr. Watson
  A Scandal In Bohemia                         Sherlock Holmes                   Dr. Watson
  The Red-Headed League                        Sherlock Holmes                   Dr. Watson
  A Case Of Identity                           Sherlock Holmes                   Dr. Watson
  The Five Orange Pips                         Sherlock Holmes                   Dr. Watson
  The Adventure Of The Blue Carbuncle          Sherlock Holmes                   Dr. Watson
  The Adventure Of The Speckled Band           Sherlock Holmes   

## 3. Personajes genéricos / ruido

In [5]:
total_chars = 0
noise_chars = 0
noise_by_story = {}

for story, result in data.items():
    chars = result["entities"].get("characters", [])
    total_chars += len(chars)
    noise = [c["name"] for c in chars if is_generic(c["name"])]
    noise_chars += len(noise)
    noise_by_story[story] = noise

print(f"Total personajes            : {total_chars}")
print(f"Personajes genericos/ruido  : {noise_chars} ({100*noise_chars/total_chars:.1f}%)")
print(f"Personajes con nombre propio: {total_chars - noise_chars}")

print("\nDetalle por relato:")
for story, noise in noise_by_story.items():
    if noise:
        print(f"\n  {story} ({len(noise)} de ruido):")
        for n in noise:
            print(f"    - '{n}'")

Total personajes            : 223
Personajes genericos/ruido  : 0 (0.0%)
Personajes con nombre propio: 223

Detalle por relato:


## 4. Relaciones huérfanas

Relaciones cuyo `source_name` no existe como entidad conocida del relato.
Se pierden silenciosamente en Neo4j.

In [6]:
total_rels = 0
orphan_total = 0
orphan_sources = Counter()

print("Relaciones huerfanas por relato:")
for story, result in data.items():
    entities = result["entities"]
    known = (
        {c["name"] for c in entities.get("characters", [])}
        | {l["name"] for l in entities.get("locations", [])}
        | {o["name"] for o in entities.get("objects", [])}
        | {s["title"] for s in entities.get("scenes", [])}
        | {e["name"] for e in entities.get("events", [])}
    )
    rels = result.get("relationships", [])
    total_rels += len(rels)
    orphans = [
        r for r in rels
        if r["relationship_type"] in CHAR_REL_TYPES and r["source_name"] not in known
    ]
    orphan_total += len(orphans)
    for r in orphans:
        orphan_sources[r["source_name"]] += 1
    if orphans:
        print(f"\n  {story}: {len(orphans)} huerfanas de {len(rels)}")
        for r in orphans[:3]:
            print(f"    {r['relationship_type']}: '{r['source_name']}' -> '{r['target_name']}'")

pct = 100 * orphan_total / total_rels if total_rels else 0
print(f"\n[METRICA] Huerfanas totales: {orphan_total} / {total_rels} ({pct:.1f}%)")
print("\nSources huerfanos mas frecuentes:")
for src, cnt in orphan_sources.most_common(10):
    print(f"  {cnt:3d}x  '{src}'")

Relaciones huerfanas por relato:

[METRICA] Huerfanas totales: 0 / 5815 (0.0%)

Sources huerfanos mas frecuentes:


In [7]:
orphan_chars = neo4j.execute_query("""
    MATCH (c:Character)
    WHERE NOT (c)-[:APPEARS_IN]->(:Story)
    RETURN c.name AS name
    ORDER BY name
""")
print(f"Personajes en Neo4j sin APPEARS_IN: {len(orphan_chars)}")
for row in orphan_chars[:20]:
    print(f"  '{row['name']}'") 

Personajes en Neo4j sin APPEARS_IN: 0


## 5. Tipos de relación fuera del schema

In [8]:
out_of_schema = Counter()
for result in data.values():
    for rel in result.get("relationships", []):
        rt = rel["relationship_type"]
        if rt not in VALID_REL_TYPES:
            out_of_schema[rt] += 1

if out_of_schema:
    print("Tipos no reconocidos (se pierden en Neo4j):")
    for rt, cnt in out_of_schema.most_common():
        print(f"  {cnt:4d}x  {rt}")
    print(f"\n[METRICA] Total relaciones perdidas: {sum(out_of_schema.values())}")
else:
    print("[OK] Todos los tipos de relacion estan en el schema.")

[OK] Todos los tipos de relacion estan en el schema.


## 6. Cobertura de cadenas de deducción

In [9]:
print(f"{'Relato':45s}  {'Deds':>5}  {'LEADS_TO':>8}  {'BASED_ON':>8}  {'% conn':>7}")
print("-" * 80)

total_deds = 0
total_lt = 0
total_bo = 0

for story, result in data.items():
    n_ded = len(result["entities"].get("deductions", []))
    rels = result.get("relationships", [])
    lt = sum(1 for r in rels if r["relationship_type"] == "LEADS_TO")
    bo = sum(1 for r in rels if r["relationship_type"] == "BASED_ON")
    pct = 100 * (lt + bo) / n_ded if n_ded else 0
    total_deds += n_ded
    total_lt += lt
    total_bo += bo
    print(f"  {story:43s}  {n_ded:5d}  {lt:8d}  {bo:8d}  {pct:6.0f}%")

total_pct = 100 * (total_lt + total_bo) / total_deds if total_deds else 0
print("-" * 80)
print(f"  {'TOTAL':43s}  {total_deds:5d}  {total_lt:8d}  {total_bo:8d}  {total_pct:6.0f}%")
print(f"\n[METRICA] Deductions conectadas: {total_pct:.0f}%  (umbral: >50%)")

Relato                                          Deds  LEADS_TO  BASED_ON   % conn
--------------------------------------------------------------------------------
  Silver Blaze                                    40         7        55     155%
  The Final Problem                               29         1        31     110%
  The Adventure Of The Dancing Men                38         6        40     121%
  A Scandal In Bohemia                            37         3        48     138%
  The Red-Headed League                           33         6        33     118%
  A Case Of Identity                              23         4        28     139%
  The Five Orange Pips                            23         2        29     135%
  The Adventure Of The Blue Carbuncle             42         2        45     112%
  The Adventure Of The Speckled Band              63        10        83     148%
  The Adventure Of The Copper Beeches             22         6        22     127%
-----------------

In [10]:
ded_check = neo4j.execute_query("""
    MATCH (d:Deduction)
    RETURN
        count(d) AS total,
        count(DISTINCT CASE WHEN (d)-[:LEADS_TO|BASED_ON]-() THEN d END) AS connected
""")
row = ded_check[0]
pct = 100 * row['connected'] / row['total'] if row['total'] else 0
print(f"Deductions en Neo4j          : {row['total']}")
print(f"Con LEADS_TO o BASED_ON      : {row['connected']} ({pct:.0f}%)")

Deductions en Neo4j          : 350
Con LEADS_TO o BASED_ON      : 186 (53%)


## 7. Consistencia de títulos de relatos

In [11]:
print("Titulos en JSON:")
title_issues = 0
for title in data.keys():
    expected = title.title()
    if title != expected:
        title_issues += 1
        print(f"  [CAPS] '{title}'  ->  deberia ser '{expected}'")
    else:
        print(f"  [OK]   '{title}'")

print(f"\n[METRICA] Titulos con formato incorrecto: {title_issues}/{len(data)}")

stories_neo4j = neo4j.execute_query(
    "MATCH (s:Story) RETURN s.title AS title ORDER BY title"
)
print("\nTitulos en Neo4j:")
for row in stories_neo4j:
    t = row['title']
    flag = "OK  " if t == t.title() else "CAPS"
    print(f"  [{flag}] '{t}'")

Titulos en JSON:
  [OK]   'Silver Blaze'
  [OK]   'The Final Problem'
  [OK]   'The Adventure Of The Dancing Men'
  [OK]   'A Scandal In Bohemia'
  [OK]   'The Red-Headed League'
  [OK]   'A Case Of Identity'
  [OK]   'The Five Orange Pips'
  [OK]   'The Adventure Of The Blue Carbuncle'
  [OK]   'The Adventure Of The Speckled Band'
  [OK]   'The Adventure Of The Copper Beeches'

[METRICA] Titulos con formato incorrecto: 0/10

Titulos en Neo4j:
  [OK  ] 'A Case Of Identity'
  [OK  ] 'A Scandal In Bohemia'
  [OK  ] 'Silver Blaze'
  [OK  ] 'The Adventure Of The Blue Carbuncle'
  [OK  ] 'The Adventure Of The Copper Beeches'
  [OK  ] 'The Adventure Of The Dancing Men'
  [OK  ] 'The Adventure Of The Speckled Band'
  [OK  ] 'The Final Problem'
  [OK  ] 'The Five Orange Pips'
  [OK  ] 'The Red-Headed League'


## 8. Distribución de objetos y eventos

In [12]:
all_objects = []
all_events = []
for result in data.values():
    for o in result["entities"].get("objects", []):
        all_objects.append(o["name"])
    for e in result["entities"].get("events", []):
        all_events.append(e["name"])

obj_counts = Counter(all_objects)
evt_counts = Counter(all_events)
print(f"Objects: {len(all_objects)} extraidos, {len(obj_counts)} unicos")
print(f"Events : {len(all_events)} extraidos, {len(evt_counts)} unicos")

print("\nObjetos en 3+ relatos (candidatos a nodo compartido):")
for name, cnt in obj_counts.most_common(20):
    if cnt >= 3:
        print(f"  {cnt}x  '{name}'")

print("\nVariantes de arma de fuego (candidatos a fusion):")
gun_kw = ["revolver", "pistol", "gun", "rifle"]
gun_variants = sorted(n for n in obj_counts if any(w in n.lower() for w in gun_kw))
for v in gun_variants:
    print(f"  {obj_counts[v]}x  '{v}'")

Objects: 642 extraidos, 562 unicos
Events : 722 extraidos, 720 unicos

Objetos en 3+ relatos (candidatos a nodo compartido):
  6x  'revolver'
  5x  'door'
  5x  'note'
  4x  'paper'
  4x  'letters'
  4x  'letter'
  3x  'lantern'
  3x  'wire'
  3x  'stick'
  3x  'papers'
  3x  'hansom'
  3x  'pistol'
  3x  'telegram'
  3x  'dog-cart'
  3x  'advertisement'

Variantes de arma de fuego (candidatos a fusion):
  1x  'air-guns'
  1x  'army revolver'
  1x  'gun'
  3x  'pistol'
  6x  'revolver'


In [13]:
orphan_obj = neo4j.execute_query(
    "MATCH (o:Object) WHERE NOT (o)-[]-() RETURN count(o) AS cnt"
)
total_obj = neo4j.execute_query("MATCH (o:Object) RETURN count(o) AS cnt")
print(f"Objects en Neo4j             : {total_obj[0]['cnt']}")
print(f"Objects sin ninguna relacion : {orphan_obj[0]['cnt']}")

Objects en Neo4j             : 562
Objects sin ninguna relacion : 0


## 9. Resumen ejecutivo

Panel de estado con todas las métricas. Volver a ejecutar tras cada corrección para
verificar el progreso.

In [14]:
stats = neo4j.get_stats()

# Nodos Holmes / Watson
n_holmes = len(neo4j.execute_query(
    "MATCH (c:Character) WHERE toLower(c.name) CONTAINS 'holmes' RETURN c"
))
n_watson = len(neo4j.execute_query(
    "MATCH (c:Character) WHERE toLower(c.name) CONTAINS 'watson' RETURN c"
))

# Ruido
total_chars_n = sum(len(r["entities"].get("characters", [])) for r in data.values())
noise_n = sum(
    1 for r in data.values()
    for c in r["entities"].get("characters", [])
    if is_generic(c["name"])
)

# Huerfanas
total_rels_n = sum(len(r.get("relationships", [])) for r in data.values())
orphan_n = sum(
    1 for result in data.values()
    for r in result.get("relationships", [])
    if r["relationship_type"] in CHAR_REL_TYPES
    and r["source_name"] not in (
        {c["name"] for c in result["entities"].get("characters", [])}
        | {l["name"] for l in result["entities"].get("locations", [])}
        | {o["name"] for o in result["entities"].get("objects", [])}
    )
)

# Fuera de schema
out_schema_n = sum(
    1 for r in data.values()
    for rel in r.get("relationships", [])
    if rel["relationship_type"] not in VALID_REL_TYPES
)

# Deducciones
total_deds_n = sum(len(r["entities"].get("deductions", [])) for r in data.values())
conn_deds_n = sum(
    1 for r in data.values()
    for rel in r.get("relationships", [])
    if rel["relationship_type"] in {"LEADS_TO", "BASED_ON"}
)

# Titulos
title_issues_n = sum(1 for t in data.keys() if t != t.title())

checks = [
    (n_holmes == 1,
     "P1 Fragmentacion Holmes",
     f"{n_holmes} nodos en Neo4j (esperado: 1)",
     "CRITICO"),
    # P2: 2 nodos es correcto — Dr. Watson (detective) y Mrs. Watson (esposa) son personas distintas
    (n_watson <= 2,
     "P2 Fragmentacion Watson",
     f"{n_watson} nodos en Neo4j (esperado: <=2: Dr. Watson + Mrs. Watson)",
     "CRITICO"),
    (noise_n / total_chars_n < 0.15,
     "P3 Personajes ruido",
     f"{noise_n}/{total_chars_n} = {100*noise_n/total_chars_n:.0f}%  (umbral: <15%)",
     "ALTO"),
    (orphan_n / total_rels_n < 0.05,
     "P4 Relaciones huerfanas",
     f"{orphan_n}/{total_rels_n} = {100*orphan_n/total_rels_n:.0f}%  (umbral: <5%)",
     "ALTO"),
    (out_schema_n == 0,
     "P5 Rel. fuera de schema",
     f"{out_schema_n} relaciones perdidas",
     "MEDIO"),
    (conn_deds_n / total_deds_n > 0.50 if total_deds_n else False,
     "P6 Cadenas deduccion",
     f"{conn_deds_n}/{total_deds_n} = {100*conn_deds_n/total_deds_n:.0f}%  (umbral: >50%)",
     "MEDIO"),
    (title_issues_n == 0,
     "P7 Titulos inconsistentes",
     f"{title_issues_n}/{len(data)} con formato incorrecto",
     "BAJO"),
]

print("=" * 70)
print("PANEL DE AUDITORIA")
print("=" * 70)
print("\nNodos en Neo4j:")
for label, cnt in sorted(stats.items(), key=lambda x: -x[1]):
    print(f"  {label:15s}: {cnt}")
print("\nChecks de calidad:")
print("-" * 70)
for ok, name, detail, severity in checks:
    icon = "[OK  ]" if ok else "[FAIL]"
    print(f"  {icon} [{severity:6s}]  {name:25s}  {detail}")
failed = sum(1 for ok, *_ in checks if not ok)
print("-" * 70)
print(f"  Checks fallidos: {failed}/{len(checks)}")
print("=" * 70)

PANEL DE AUDITORIA

Nodos en Neo4j:
  Event          : 722
  Object         : 562
  Chunk          : 350
  Deduction      : 350
  Crime          : 317
  Scene          : 303
  Location       : 261
  Character      : 186
  Story          : 10

Checks de calidad:
----------------------------------------------------------------------
  [OK  ] [CRITICO]  P1 Fragmentacion Holmes    1 nodos en Neo4j (esperado: 1)
  [OK  ] [CRITICO]  P2 Fragmentacion Watson    2 nodos en Neo4j (esperado: <=2: Dr. Watson + Mrs. Watson)
  [OK  ] [ALTO  ]  P3 Personajes ruido        0/223 = 0%  (umbral: <15%)
  [OK  ] [ALTO  ]  P4 Relaciones huerfanas    17/5815 = 0%  (umbral: <5%)
  [OK  ] [MEDIO ]  P5 Rel. fuera de schema    0 relaciones perdidas
  [OK  ] [MEDIO ]  P6 Cadenas deduccion       461/350 = 132%  (umbral: >50%)
  [OK  ] [BAJO  ]  P7 Titulos inconsistentes  0/10 con formato incorrecto
----------------------------------------------------------------------
  Checks fallidos: 0/7


In [15]:
# P3 — Detalle de los personajes que el notebook detecta como ruido
# Útil para afinar _GENERIC_REFS en entity_extractor.py
print("Personajes detectados como ruido por is_generic() en el JSON actual:")
print(f"{'Relato':45s}  Nombre")
print("-" * 80)
noise_names = []
for story, result in data.items():
    for c in result["entities"].get("characters", []):
        if is_generic(c["name"]):
            noise_names.append(c["name"])
            print(f"  {story:43s}  '{c['name']}'")

from collections import Counter
print(f"\nTotal: {len(noise_names)}")
print("\nMás frecuentes (aparecen en varios relatos):")
for name, cnt in Counter(noise_names).most_common(15):
    if cnt > 1:
        print(f"  {cnt}x  '{name}'")

Personajes detectados como ruido por is_generic() en el JSON actual:
Relato                                         Nombre
--------------------------------------------------------------------------------

Total: 0

Más frecuentes (aparecen en varios relatos):


In [16]:
neo4j.close()
print("Conexion cerrada.")

Conexion cerrada.
